<a href="https://colab.research.google.com/github/manuflog/contextuality-obstructions/blob/main/ibm_holonomy_experiment_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Contextual holonomy on IBM hardware — two loops, two invariants
**A:** WF/MUB loop, interferometric. Targets: strands **+45/−45°**, loop **+1**.
**B:** Peres–Mermin via qkernel protocol. Targets: signs (+,+,+,+,+,**−**), S above NC bound.
**One-time setup (10 s):** left sidebar → key icon (*Secrets*) → *Add new secret* →
name `IBM_QUANTUM_TOKEN`, value = your API key → toggle *Notebook access* ON.
After that this notebook never asks for the key again (falls back to a prompt if the secret
is absent). Instance CRN is already baked in below. Run top to bottom; Aer gate blocks hardware.

In [1]:
!pip install -q qiskit qiskit-aer qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.3/111.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.2/224.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 10.9 MB/s eta 0:00:00


In [2]:
import numpy as np
H=np.array([[1,1],[1,-1]])/np.sqrt(2); S=np.diag([1,1j])
BASES=[np.eye(2), H, S@H]
def hadamard_test_plan():
    plan=[]
    for k in (0,1):
        for a in range(3):
            U=BASES[a].conj().T @ BASES[(a+1)%3]
            for part in ("re","im"):
                plan.append(dict(strand=k,step=a,part=part,prep=k,unitary=U))
    return plan
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import UnitaryGate
def htest_circuit(p):
    qc=QuantumCircuit(2,1)
    if p["prep"]: qc.x(1)
    qc.h(0)
    if p["part"]=="im": qc.sdg(0)
    qc.append(UnitaryGate(p["unitary"]).control(1),[0,1])
    qc.h(0); qc.measure(0,0); return qc
PLAN=hadamard_test_plan(); print(len(PLAN),"circuits")

12 circuits


In [3]:
from qiskit_aer import AerSimulator
sim=AerSimulator(); SHOTS=20000
wrap=lambda a:(a+np.pi)%(2*np.pi)-np.pi
def analyze(getcounts):
    ov={}
    for p in PLAN:
        c=getcounts(p)
        ov.setdefault((p["strand"],p["step"]),{})[p["part"]]=(c.get("0",0)-c.get("1",0))/sum(c.values())
    out={}
    for k in (0,1):
        z=[ov[(k,s)]["re"]+1j*ov[(k,s)]["im"] for s in range(3)]
        out[k]=(np.degrees(wrap(sum(np.angle(x) for x in z))), np.prod(z))
    return out
res=analyze(lambda p: sim.run(transpile(htest_circuit(p),sim),shots=SHOTS).result().get_counts())
ok=abs(res[0][0]-45)<3 and abs(res[1][0]+45)<3
print({k:f"{v[0]:+.2f} deg" for k,v in res.items()})
print("PASS - proceed to hardware" if ok else "FAIL - do NOT run hardware")
assert ok

{0: '+44.80 deg', 1: '-46.58 deg'}
PASS - proceed to hardware


In [4]:
INSTANCE = "crn:v1:bluemix:public:quantum-computing:us-east:a/1e742358669f410787a8210dd5965067:dbb158e1-ca83-49e8-81cd-d286bae718dd::"
def get_token():
    try:
        from google.colab import userdata
        t = userdata.get("IBM_QUANTUM_TOKEN")
        if t: return t
    except Exception:
        pass
    from getpass import getpass
    return getpass("IBM Cloud API key: ")
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Batch
QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token=get_token(),
                                  instance=INSTANCE, set_as_default=True, overwrite=True)
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=3)
print("backend:", backend.name)   # also covers pm_protocol.py (reads the saved account)

backend: ibm_fez


In [5]:
HW_SHOTS=8192
tqcs=[transpile(htest_circuit(p),backend,optimization_level=3) for p in PLAN]
with Batch(backend=backend):
    job=Sampler().run(tqcs,shots=HW_SHOTS)
print("job:",job.job_id())
res_hw=job.result()
def cnt(i):
    d=res_hw[i].data
    reg=getattr(d,"c",None) or list(d.values())[0]
    return reg.get_counts()
hw=analyze(lambda p: cnt(PLAN.index(p)))
for k in (0,1):
    print(f"strand{k}: {hw[k][0]:+.2f} deg  (target {'+45' if k==0 else '-45'})")
loop=hw[0][1]*hw[1][1]
print(f"loop holonomy phase: {np.degrees(np.angle(loop)):+.2f} deg (target 0)")

job: d986q62f47jc73a7hm2g
strand0: +45.88 deg  (target +45)
strand1: -48.81 deg  (target -45)
loop holonomy phase: -2.93 deg (target 0)


## Experiment B — Peres–Mermin loop (qkernel exported protocol)
Auth is already saved by the cell above. Expected: (+,+,+,+,+,−) and S above the NC bound.
Paste both experiments' outputs back to Claude → hardware_registry + Paper C §4 + scorecard.

In [6]:
%%writefile pm_protocol.py
"""Runnable IBM contextuality protocol, exported by qkernel export-circuit.
Sequential non-destructive (ancilla Hadamard-test) measurement of each context's
three commuting observables -- a genuine, noise-sensitive statistic (NOT the
algebraically-pinned single-joint-measurement, which returns the ideal value on any
data).  2 data qubits + 1 ancilla; DD + twirling; pinned to one low-error triple.
"""
import math
from qiskit import QuantumCircuit, transpile, QuantumRegister, ClassicalRegister
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

CONTEXTS = {'C0': (['XI', 'IX', 'XX'], 1), 'C1': (['IZ', 'ZI', 'ZZ'], 1), 'C2': (['XZ', 'ZX', 'YY'], 1), 'C3': (['XI', 'IZ', 'XZ'], 1), 'C4': (['IX', 'ZI', 'ZX'], 1), 'C5': (['XX', 'ZZ', 'YY'], -1)}
NC_BOUND = 5.0
IDEAL = 6.0

def _ctrl(qc, anc, q, P):
    if P == 'I': return
    if P == 'X': qc.cx(anc, q)
    elif P == 'Z': qc.cz(anc, q)
    elif P == 'Y': qc.sdg(q); qc.cx(anc, q); qc.s(q)

def _measure_obs(qc, data, anc, cbit, label):
    qc.h(anc); _ctrl(qc, anc, data[0], label[0]); _ctrl(qc, anc, data[1], label[1])
    qc.h(anc); qc.measure(anc, cbit); qc.reset(anc)

def build(name):
    data = QuantumRegister(2, 'd'); anc = QuantumRegister(1, 'a'); c = ClassicalRegister(3, 'c')
    qc = QuantumCircuit(data, anc, c)
    labels, _ = CONTEXTS[name]
    for j, lab in enumerate(labels):
        _measure_obs(qc, data, anc[0], c[j], lab)
    return qc

def product_expectation(name, counts):
    # shot-by-shot product of the THREE separately-measured ancilla parities
    tot = sum(counts.values()); acc = 0.0
    for bits, cnt in counts.items():
        b = bits.replace(' ', ''); o = [(-1)**int(b[-1-j]) for j in range(3)]
        acc += o[0]*o[1]*o[2] * cnt
    return acc / tot

def _se(counts):
    tot = sum(counts.values()); even = sum(c for b,c in counts.items() if b.replace(' ','').count('1')%2==0)
    p = even/tot; return math.sqrt(max(4*p*(1-p)/tot, 0.0))

def _best_triple(backend):
    t = backend.target; edges = set(t.build_coupling_map().get_edges()); nq = t.num_qubits
    def ro(q):
        try: return t['measure'][(q,)].error or 0.05
        except Exception: return 0.05
    tw = [g for g in ['cz','ecr','cx'] if g in t.operation_names]
    def ee(a,b):
        for g in tw:
            for pr in [(a,b),(b,a)]:
                try:
                    e = t[g][pr].error
                    if e is not None: return e
                except Exception: pass
        return 0.05
    nbr = {q:set() for q in range(nq)}
    for a,b in edges: nbr[a].add(b); nbr[b].add(a)
    best=None; bs=1e9
    for anc in range(nq):
        nb=list(nbr[anc])
        for i in range(len(nb)):
            for j in range(i+1,len(nb)):
                d0,d1=nb[i],nb[j]; s=ro(anc)+ro(d0)+ro(d1)+ee(anc,d0)+ee(anc,d1)
                if s<bs: bs=s; best=[d0,d1,anc]
    return best

if __name__ == '__main__':
    import os
    service = QiskitRuntimeService(token=os.environ.get('QISKIT_IBM_TOKEN'))
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=3)
    print('Backend:', backend.name)
    layout = _best_triple(backend); print('Qubit triple [d0,d1,anc]:', layout)
    sampler = Sampler(mode=backend)
    sampler.options.dynamical_decoupling.enable = True
    sampler.options.dynamical_decoupling.sequence_type = 'XpXm'
    sampler.options.twirling.enable_gates = True
    sampler.options.twirling.enable_measure = True
    S = 0.0; Svar = 0.0
    for name, (obs, sgn) in CONTEXTS.items():
        qc = transpile(build(name), backend, initial_layout=layout, optimization_level=1)
        counts = sampler.run([qc], shots=8192).result()[0].data.c.get_counts()
        val = product_expectation(name, counts); se = _se(counts); S += sgn*val; Svar += se*se
        print(f'{name} {obs}: <prod>={val:+.3f} +/- {se:.3f} (ideal {sgn:+d})')
    SE = Svar**0.5
    print(f'\nS = {S:.4f} +/- {SE:.4f}   NC bound = {NC_BOUND}   ideal = {IDEAL}')
    print(f'({(S-NC_BOUND)/SE:.1f} sigma above bound)' if SE>0 else '')
    print('CONTEXTUALITY CERTIFIED' if S - 2*SE > NC_BOUND else 'no clean violation - check calibration/backend')


Writing pm_protocol.py


In [7]:
!python pm_protocol.py

Backend: ibm_fez
Qubit triple [d0,d1,anc]: [21, 23, 22]
C0 ['XI', 'IX', 'XX']: <prod>=+0.750 +/- 0.007 (ideal +1)
C1 ['IZ', 'ZI', 'ZZ']: <prod>=+0.835 +/- 0.006 (ideal +1)
C2 ['XZ', 'ZX', 'YY']: <prod>=+0.722 +/- 0.008 (ideal +1)
C3 ['XI', 'IZ', 'XZ']: <prod>=+0.780 +/- 0.007 (ideal +1)
C4 ['IX', 'ZI', 'ZX']: <prod>=+0.824 +/- 0.006 (ideal +1)
C5 ['XX', 'ZZ', 'YY']: <prod>=-0.700 +/- 0.008 (ideal -1)

S = 4.6125 +/- 0.0173   NC bound = 5.0   ideal = 6.0
(-22.5 sigma above bound)
no clean violation - check calibration/backend
